# Notebook 01c: Leakage Audit
This notebook validates the feature set against the audit-approved whitelist. We ensure that Problem A (Severity) uses only pre-circumstance features, while Problem B (Category) is allowed to use retrospective data.

In [ ]:
import pandas as pd
import yaml

with open('configs/main_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

with open('configs/feature_whitelist.yaml', 'r') as f:
    whitelist = yaml.safe_load(f)

df = pd.read_parquet(config['paths']['raw_data'])
print(f"Loaded {len(df)} records with {len(df.columns)} columns.")

## 1. Whitelist Verification

In [ ]:
prob_a_cols = [item['feature'] for item in whitelist['problem_a_whitelist']]
prob_b_cols = [item['feature'] for item in whitelist['problem_b_whitelist']] + prob_a_cols

missing_a = [c for c in prob_a_cols if c not in df.columns]
missing_b = [c for c in prob_b_cols if c not in df.columns]

if missing_a:
    print(f"WARNING: Missing Problem A whitelist columns in data: {missing_a}")
if missing_b:
    print(f"WARNING: Missing Problem B whitelist columns in data: {missing_b}")

available_a = [c for c in prob_a_cols if c in df.columns]
available_b = [c for c in prob_b_cols if c in df.columns]

print(f"Problem A: {len(available_a)} whitelist columns available.")
print(f"Problem B: {len(available_b)} whitelist columns available.")

## 2. Leakage Test (Automated Check)
We explicitly verify that no 'Events' or 'Assessments' columns (other than those whitelisted for B) are present in the Problem A feature set.

In [ ]:
leakage_keywords = ['Events', 'Assessments', 'Narrative', 'Synopsis', 'Injury', 'Result', 'Damage']
potential_leakage = []

for col in available_a:
    if any(k in col for k in leakage_keywords):
        # Check if it was explicitly justified
        justified = False
        for item in whitelist['problem_a_whitelist']:
            if item['feature'] == col:
                justified = True
                break
        if not justified:
            potential_leakage.append(col)

if potential_leakage:
    print(f"CRITICAL: Potential un-justified leakage columns in Problem A set: {potential_leakage}")
else:
    print("Problem A leakage check passed (no un-justified keywords found).")

## 3. Save Whitelisted Feature Data

In [ ]:
# Filter data to whitelisted columns and save for modeling
all_relevant_cols = list(set(available_a + available_b + ['acn_num_ACN']))
df_filtered = df[all_relevant_cols]
df_filtered.to_parquet('data/processed/whitelisted_features.parquet')
print(f"Saved filtered data with {len(df_filtered.columns)} columns to data/processed/whitelisted_features.parquet")